# 01_agentcard_sendmessage

Stage 1 of the `codes/a2a/` track: **Agent Card discovery + JSON-RPC `SendMessage`**
on a **stdlib-only** local server (A2A 1.0.1 wire names).

Notebook path: `codes/a2a/01_agentcard_sendmessage/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). **No new pip deps** —
unlike `codes/mcp/`, this lab deliberately shows the wire instead of hiding it
behind an SDK. Shared `codes/requirements.txt` is unchanged.

Run the next cell first to bind `ROOT` and start the lab server in a background
thread. Edit only under `# answer`. Pre-filled setup stays as-is.

Architecture to keep in your head:

`GET /.well-known/agent-card.json` → `POST /a2a/jsonrpc` `SendMessage` (+ `A2A-Version: 1.0`) → task states `WORKING` / `INPUT_REQUIRED` / `COMPLETED`

Contrast with MCP: MCP exposes **tools for one agent**; A2A delegates **tasks between agents** via cards and a task lifecycle.


In [1]:
from pathlib import Path
import sys
import threading
import time

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_agentcard_sendmessage" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "a2a" / "01_agentcard_sendmessage"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/a2a/01_agentcard_sendmessage")

ROOT = lab_root()
sys.path.insert(0, str(ROOT))
print(ROOT)

from server import serve, DEFAULT_HOST, DEFAULT_PORT, A2A_VERSION
from client import fetch_card, rpc, new_user_message, wait_for_server

BASE = f"http://{DEFAULT_HOST}:{DEFAULT_PORT}"
_server = serve(DEFAULT_HOST, DEFAULT_PORT)
_thread = threading.Thread(target=_server.serve_forever, daemon=True)
_thread.start()
wait_for_server(BASE)
print("server ready →", BASE, "A2A-Version", A2A_VERSION)


/workspace/AEFS-Notes/codes/a2a/01_agentcard_sendmessage
server ready → http://127.0.0.1:41242 A2A-Version 1.0


## 1. Read the Agent Card shape

Fetch `/.well-known/agent-card.json`. Print the agent `name`, every skill `id`,
and each `supportedInterfaces` entry as `protocolBinding protocolVersion @ url`.


In [2]:
# answer
card = fetch_card(BASE)
print("name →", card["name"])
print("skills →", [s["id"] for s in card["skills"]])
for iface in card["supportedInterfaces"]:
    print(
        f"interface → {iface['protocolBinding']} "
        f"{iface['protocolVersion']} @ {iface['url']}"
    )


name → writer-agent
skills → ['draft_summary']
interface → JSONRPC 1.0 @ http://127.0.0.1:41242/a2a/jsonrpc


## 2. SendMessage without targetLength → INPUT_REQUIRED

Build a `ROLE_USER` message with only a text part (no `data.targetLength`).
Call `SendMessage`. Print `task.id`, `task.contextId`, `status.state`, and the
agent's question text from `status.message.parts[0].text`.


In [3]:
# answer
first = new_user_message([{"text": "Summarize MCP vs A2A for a lab notebook."}])
reply = rpc(BASE, "SendMessage", {"message": first}, rpc_id=1)
assert "error" not in reply, reply
task = reply["result"]["task"]
print("task id    →", task["id"])
print("context id →", task["contextId"])
print("state      →", task["status"]["state"])
ask = task["status"]["message"]["parts"][0]["text"]
print("agent asks →", ask)
assert task["status"]["state"] == "TASK_STATE_INPUT_REQUIRED"
TASK_ID = task["id"]
CONTEXT_ID = task["contextId"]


task id    → task_4a276bfeae
context id → 32aff5b3-2f71-4402-bcbc-3467f907e338
state      → TASK_STATE_INPUT_REQUIRED
agent asks → Please specify targetLength as a data part (e.g. {"data": {"targetLength": "3 paragraphs"}}).


## 3. Follow-up on the same task → COMPLETED + artifact

Send a second `SendMessage` with the same `taskId` / `contextId` and a data part
`{"targetLength": "3 paragraphs"}`. Print the final state and the artifact text.


In [4]:
# answer
follow = new_user_message(
    [{"data": {"targetLength": "3 paragraphs"}, "mediaType": "application/json"}],
    task_id=TASK_ID,
    context_id=CONTEXT_ID,
)
reply2 = rpc(BASE, "SendMessage", {"message": follow}, rpc_id=2)
assert "error" not in reply2, reply2
task2 = reply2["result"]["task"]
print("state    →", task2["status"]["state"])
art = task2["artifacts"][0]
print("artifact →", art["name"])
print("content  →", art["parts"][0]["text"])
assert task2["status"]["state"] == "TASK_STATE_COMPLETED"


state    → TASK_STATE_COMPLETED
artifact → summary
content  → [writer-agent] 3 paragraphs summary of provided source: topic identified, key points extracted, conclusion drafted.


## 4. Missing `A2A-Version` → `-32009`

Call `SendMessage` with `version=None` so the client omits the header. Print the
JSON-RPC error `code` and `message`. A 1.0 peer must reject this.


In [5]:
# answer
bad = rpc(
    BASE,
    "SendMessage",
    {"message": new_user_message([{"text": "ping"}])},
    rpc_id=3,
    version=None,
)
err = bad["error"]
print("code    →", err["code"])
print("message →", err["message"])
assert err["code"] == -32009


code    → -32009
message → VersionNotSupportedError


## 5. MCP vs A2A — write it once

In one short paragraph (print a string), contrast MCP and A2A: who talks to whom,
what is discovered, and what the unit of work is.


In [6]:
# answer
print(
    "MCP connects one agent/host to tool servers (tools/list → tools/call). "
    "A2A connects agents to agents: discover an Agent Card, then SendMessage "
    "to drive a task through TASK_STATE_* until an artifact or a terminal failure."
)


MCP connects one agent/host to tool servers (tools/list → tools/call). A2A connects agents to agents: discover an Agent Card, then SendMessage to drive a task through TASK_STATE_* until an artifact or a terminal failure.


## Cleanup

Stop the background server when you are done exploring.


In [7]:
_server.shutdown()
print("server stopped")


server stopped
